# CoCa classifier validation
Run all is safe: this notebook validates the real shared-Drive candidate, real CoCa weights, native preprocessing, frozen-encoder behavior, and checkpoint guards. It never creates or starts the six formal runs.

In [ ]:
from pathlib import Path
EXPECTED_GIT_COMMIT = "1aff37842730ee852130d094f11647676d56fc17"
REPO_URL = "https://github.com/sfczaa/ddpm-derm-augmentation.git"
RUN_VERSION = "v1"
CHECKPOINT_FORMAT = "frozen_backbone_head_only_v1"
MAX_COCA_CHECKPOINT_BYTES = 100 * 1024 * 1024
SHARED_PROJECT_DIR = Path("/content/drive/MyDrive/ddpm-derm-augmentation")
SHARED_RUN_ROOT = Path("/content/drive/MyDrive/ddpm-derm-coca-runs")
COCA_ROOT = SHARED_RUN_ROOT / "sqrt_balanced_seed0_v1" / "coca_classifier" / RUN_VERSION
VALIDATION_RUNS_ROOT = COCA_ROOT / "validation_runs"
VALIDATION_RECORD = COCA_ROOT / "validation_record.json"
SHARED_ROOT_SENTINEL = SHARED_RUN_ROOT / ".coca_shared_root.json"
CANDIDATE_MANIFEST = SHARED_PROJECT_DIR / "outputs" / "exploratory_balanced_ddpm" / "sqrt_balanced_seed0_v1" / "candidate_synthetic_df" / "epoch0100_seed0" / "synthetic_df.csv"
EXPECTED_CANDIDATE_SHA256 = "9ef9b44e404f74aab8211f4e7d123da3258ba8ba4e3004a4147d1761ed343b34"
DRIVE_FOLDER_ID = None  # Optional: paste the actual shared folder ID when available.
assert len(EXPECTED_GIT_COMMIT) == 40 and EXPECTED_GIT_COMMIT != "REPLACE_AFTER_PUSH", "Pin the reviewed pushed commit before validation"

## Phase 0 CHECK — Drive, GPU, exact runtime-local checkout, dependencies

In [ ]:
import os, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")
assert SHARED_PROJECT_DIR.is_dir(), f"missing shared project shortcut: {SHARED_PROJECT_DIR}"
assert SHARED_RUN_ROOT.is_dir(), f"missing shared run shortcut; do not create a private replacement: {SHARED_RUN_ROOT}"
subprocess.run(["nvidia-smi"], check=True)
CODE_DIR = Path("/content/ddpm-coca-code")
assert not CODE_DIR.exists(), f"fresh runtime required: {CODE_DIR}"
subprocess.run(["git", "clone", REPO_URL, str(CODE_DIR)], check=True)
subprocess.run(["git", "-C", str(CODE_DIR), "checkout", "--detach", EXPECTED_GIT_COMMIT], check=True)
commit = subprocess.check_output(["git", "-C", str(CODE_DIR), "rev-parse", "HEAD"], text=True).strip()
status = subprocess.check_output(["git", "-C", str(CODE_DIR), "status", "--short"], text=True).strip()
assert commit == EXPECTED_GIT_COMMIT and not status
os.environ["HF_HOME"] = "/content/hf-cache"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "open_clip_torch==3.3.0", "pandas>=2.0", "pillow>=9.0"], check=True)
sys.path.insert(0, str(CODE_DIR / "src"))
from importlib.metadata import version
import torch
assert torch.cuda.is_available(), "CUDA GPU required"
assert version("open_clip_torch") == "3.3.0"
print({"git_commit": commit, "torch": torch.__version__, "open_clip_torch": version("open_clip_torch"), "gpu": torch.cuda.get_device_name(0)})

## Phase 0 CHECK — shared-root identity and Drive probes

In [ ]:
import json
from ddpm_derm import coca_run
resolved_run_root = coca_run.require_existing_shared_root(SHARED_RUN_ROOT)
drive_probe = coca_run.probe_shared_drive(resolved_run_root)
sentinel = coca_run.create_or_validate_sentinel(SHARED_ROOT_SENTINEL, shortcut_alias=SHARED_RUN_ROOT.name, resolved_path=str(resolved_run_root), drive_folder_id=DRIVE_FOLDER_ID, run_version=RUN_VERSION)
assert sentinel["shortcut_alias"] == "ddpm-derm-coca-runs"
assert sentinel["resolved_path"] == str(resolved_run_root)
assert sentinel["run_version"] == RUN_VERSION
if DRIVE_FOLDER_ID is not None: assert sentinel["drive_folder_id"] == DRIVE_FOLDER_ID
coca_run.ensure_tree(SHARED_RUN_ROOT, COCA_ROOT.relative_to(SHARED_RUN_ROOT))
coca_run.ensure_tree(SHARED_RUN_ROOT, VALIDATION_RUNS_ROOT.relative_to(SHARED_RUN_ROOT))
print(json.dumps(sentinel, indent=2))

## Phase 1 CHECK — fixed split, C1/C4 composition, real candidate hash and provenance

In [ ]:
import hashlib, shutil
import pandas as pd
def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""): digest.update(chunk)
    return digest.hexdigest()
assert CANDIDATE_MANIFEST.is_file(), f"candidate manifest missing after Drive mount: {CANDIDATE_MANIFEST}"
candidate_hash = sha256(CANDIDATE_MANIFEST)
assert candidate_hash == EXPECTED_CANDIDATE_SHA256, (candidate_hash, EXPECTED_CANDIDATE_SHA256)
LOCAL_DATA_DIR = Path("/content/ham10000-data")
assert not LOCAL_DATA_DIR.exists()
shutil.copytree(SHARED_PROJECT_DIR / "data", LOCAL_DATA_DIR)
os.environ["DDPM_DERM_DATA_DIR"] = str(LOCAL_DATA_DIR)
frames = {split: pd.read_csv(LOCAL_DATA_DIR / "manifests" / f"{split}.csv") for split in ("train", "val", "test")}
assert {key: len(value) for key, value in frames.items()} == {"train": 6995, "val": 1510, "test": 1510}
assert [int((frames[s]["dx"] == "df").sum()) for s in ("train", "val", "test")] == [85, 14, 16]
for other in ("val", "test"):
    assert not set(frames["train"]["lesion_id"]) & set(frames[other]["lesion_id"])
    assert not set(frames["train"]["image_id"]) & set(frames[other]["image_id"])
candidate = pd.read_csv(CANDIDATE_MANIFEST)
assert len(candidate) == 500 and candidate["image_path"].nunique() == 500
assert candidate["dx"].eq("df").all() and candidate["label_idx"].eq(3).all() and candidate["source"].eq("synthetic").all()
candidate_root = CANDIDATE_MANIFEST.parent
missing_candidate_images = [p for p in candidate["image_path"] if not (candidate_root / p).is_file()]
assert not missing_candidate_images
for other in ("val", "test"):
    assert not set(candidate["image_id"]) & set(frames[other]["image_id"])
    assert not set(candidate["lesion_id"]) & set(frames[other]["lesion_id"])
ddpm_metadata_path = CANDIDATE_MANIFEST.parents[2] / "run_metadata.json"
assert ddpm_metadata_path.is_file(), f"missing DDPM provenance record: {ddpm_metadata_path}"
ddpm_metadata = json.loads(ddpm_metadata_path.read_text(encoding="utf-8"))
assert ddpm_metadata["source_split"] == "train" and ddpm_metadata["sampler_strategy"] == "sqrt_balanced"
fixed_split_identity = sha256(LOCAL_DATA_DIR / "manifests" / "train.csv")
print({"candidate_rows": len(candidate), "candidate_sha256": candidate_hash, "C1_df_rows": 585, "C4_df_rows": 85 + len(candidate), "leakage": 0})

## Phases 2–3 CHECK — real CoCa weights, native transforms, forward, freeze, optimizer

In [ ]:
import open_clip
from PIL import Image
from ddpm_derm.model import build_model, model_identity
from ddpm_derm.train_classifier import build_optimizer
assert ("coca_ViT-B-32", "laion2b_s13b_b90k") in set(map(tuple, open_clip.list_pretrained()))
model = build_model(arch="coca_vit_b32", freeze_backbone=True, coca_pretrained="laion2b_s13b_b90k").cuda()
print("native train preprocessing:", model.train_preprocess)
print("native eval preprocessing:", model.eval_preprocess)
print("actual input resolution:", model.input_resolution)
sample_paths = [LOCAL_DATA_DIR / p for p in frames["train"]["image_path"].head(2)]
batch = torch.stack([model.eval_preprocess(Image.open(path).convert("RGB")) for path in sample_paths]).cuda()
grad_flags = []
hook = model.encoder.visual.register_forward_hook(lambda module, inputs, output: grad_flags.append(torch.is_grad_enabled()))
model.train()
logits = model(batch)
hook.remove()
assert tuple(logits.shape) == (2, 7)
assert not model.encoder.training and grad_flags and not any(grad_flags)
assert all(not p.requires_grad for p in model.encoder.parameters())
assert all(p.requires_grad for p in model.head.parameters())
optimizer = build_optimizer(model, 3e-4, 1e-4)
optimized = {id(p) for group in optimizer.param_groups for p in group["params"]}
assert optimized == {id(p) for p in model.head.parameters()}
model_details = model_identity(model, "coca_vit_b32", 128)
assert model_details["open_clip_torch_version"] == "3.3.0"
print(json.dumps(model_details, indent=2))
del model, optimizer, batch, logits
torch.cuda.empty_cache()

## Phase 4 RUN — isolated one-epoch C1 and C4 smoke runs

In [ ]:
import time
from datetime import datetime, timezone
validation_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
VALIDATION_DIR = coca_run.ensure_tree(SHARED_RUN_ROOT, (VALIDATION_RUNS_ROOT / validation_id).relative_to(SHARED_RUN_ROOT))
formal_output_identity = f"{sentinel['shared_root_uuid']}:sqrt_balanced_seed0_v1:coca_classifier:v1:formal"
env = os.environ.copy(); env["PYTHONPATH"] = str(CODE_DIR / "src"); env["PYTHONUNBUFFERED"] = "1"
def run_stream(command, expect_success=True):
    started = time.monotonic(); process = subprocess.Popen(command, cwd=CODE_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    for line in process.stdout: lines.append(line); print(line, end="", flush=True)
    code = process.wait()
    if expect_success and code: raise subprocess.CalledProcessError(code, command)
    if not expect_success and not code: raise AssertionError("command unexpectedly succeeded")
    return time.monotonic() - started, "".join(lines)
common = [sys.executable, "-u", "-m", "ddpm_derm.train_classifier", "--arch", "coca_vit_b32", "--freeze-backbone", "--coca-pretrained", "laion2b_s13b_b90k", "--seed", "0", "--epochs", "1", "--batch-size", "8", "--lr", "3e-4", "--weight-decay", "1e-4", "--df-target-count", "585", "--limit", "64", "--num-workers", "2", "--output-dir", str(VALIDATION_DIR), "--run-version", RUN_VERSION, "--shared-root-uuid", sentinel["shared_root_uuid"], "--formal-output-identity", formal_output_identity, "--fixed-split-identity", fixed_split_identity, "--candidate-sha256", candidate_hash]
smoke_times = {}
commands = {}
for variant in ("C1", "C4"):
    command = common + ["--variant", variant, "--run-label", f"coca_validation_{variant}"]
    if variant == "C4": command += ["--generated-manifest", str(CANDIDATE_MANIFEST)]
    commands[variant] = command
    smoke_times[variant], output = run_stream(command)
    assert "checkpoint_saved=last.pt" in output
print("isolated smoke seconds:", smoke_times)

## Phase 5 CHECK — checkpoint reopen, resume/skip, mismatch rejection, child restore

In [ ]:
import copy
from ddpm_derm import classifier_run
from ddpm_derm.train_classifier import validate_checkpoint_payload
checkpoint_checks = {}
checkpoint_sizes = {}
validation_model = build_model(arch="coca_vit_b32", freeze_backbone=True, coca_pretrained="laion2b_s13b_b90k").cpu()
for variant in ("C1", "C4"):
    checkpoint_dir = VALIDATION_DIR / "checkpoints" / "coca_vit_b32" / f"{variant}_seed0"
    result_path = VALIDATION_DIR / "results" / "coca_vit_b32" / f"results_{variant}_seed0.json"
    last_path, best_path = checkpoint_dir / "last.pt", checkpoint_dir / "best.pt"
    assert last_path.is_file() and best_path.is_file() and result_path.is_file()
    saved = torch.load(last_path, map_location="cpu", weights_only=False)
    best_saved = torch.load(best_path, map_location="cpu", weights_only=False)
    for checkpoint_path, checkpoint in ((last_path, saved), (best_path, best_saved)):
        assert checkpoint["checkpoint_format"] == CHECKPOINT_FORMAT
        assert "head_state_dict" in checkpoint and "model_state_dict" not in checkpoint and "encoder_state_dict" not in checkpoint
        assert not any(any(token in key.lower() for token in ("encoder", "text", "caption", "decoder", "visual")) for key in checkpoint)
        validate_checkpoint_payload(checkpoint, validation_model, checkpoint["run_identity"], "coca_vit_b32")
        coca_run.checkpoint_size(checkpoint_path, arch="coca_vit_b32")
        assert checkpoint_path.stat().st_size <= MAX_COCA_CHECKPOINT_BYTES
    checkpoint_sizes[variant] = {"last_pt_bytes": last_path.stat().st_size, "best_pt_bytes": best_path.stat().st_size}
    assert saved["epoch"] == 1 and len(saved["history"]) == 1
    before = (sha256(last_path), last_path.stat().st_mtime_ns)
    _, output = run_stream(commands[variant] + ["--resume"])
    assert "[resume]" in output and "[skip]" in output
    for key, value in (("arch", "resnet18"), ("pretrained_tag", "wrong"), ("freeze_mode", "trainable"), ("preprocessing_identity", {"train": "wrong", "eval": "wrong"})):
        changed = copy.deepcopy(saved["run_identity"]); changed["model_identity"][key] = value
        try: classifier_run.require_matching_resume_identity(saved["run_identity"], changed); raise AssertionError(key)
        except ValueError: pass
    changed = copy.deepcopy(saved["run_identity"]); changed["checkpoint_format"] = "full_model_v1"
    try: classifier_run.require_matching_resume_identity(saved["run_identity"], changed); raise AssertionError("checkpoint format")
    except ValueError: pass
    wrong_payload = copy.deepcopy(saved); wrong_payload["checkpoint_format"] = "full_model_v1"
    try: validate_checkpoint_payload(wrong_payload, validation_model, saved["run_identity"], "coca_vit_b32"); raise AssertionError("payload checkpoint format")
    except ValueError: pass
    changed = copy.deepcopy(saved["run_identity"]); changed["candidate_manifest_sha256"] = "0" * 64
    try: classifier_run.require_matching_resume_identity(saved["run_identity"], changed); raise AssertionError("candidate hash")
    except ValueError: pass
    assert (sha256(last_path), last_path.stat().st_mtime_ns) == before
    checkpoint_checks[variant] = {"last_reopened": True, "best_reopened": True, "resume_skip": True, "mismatches_rejected_without_mutation": True}
child_code = "import subprocess,sys; subprocess.run(sys.argv[1:], check=True)"
subprocess.run([sys.executable, "-c", child_code] + commands["C1"] + ["--resume"], cwd=CODE_DIR, env=env, check=True)
checkpoint_checks["child_process_drive_restore"] = True
del validation_model
print(json.dumps(checkpoint_checks, indent=2))

## Phase 6 REVIEW — fixed validation record and hard stop before formal training

In [ ]:
estimated_seconds = float(sum(smoke_times.values()) / 2 * (7495 / 64) * 120)
record = {"validation_status": "VALIDATION PASSED", "formal_training_started": False, "git_commit": commit, "run_version": RUN_VERSION, "candidate_manifest_sha256": candidate_hash, "fixed_split_identity": fixed_split_identity, "shared_root_uuid": sentinel["shared_root_uuid"], "shared_root_identity": sentinel, "formal_output_identity": formal_output_identity, "model_identity": model_details, "checkpoint_format": CHECKPOINT_FORMAT, "checkpoint_sizes": checkpoint_sizes, "encoder_weights_stored": False, "dependency_versions": {"open_clip_torch": version("open_clip_torch"), "torch": torch.__version__}, "drive_probes": drive_probe, "smoke_runs": smoke_times, "checkpoint_checks": checkpoint_checks, "validation_timestamp": coca_run.utc_now(), "validation_artifact_directory": str(VALIDATION_DIR), "estimated_formal_six_run_seconds": estimated_seconds, "estimate_basis": "two 64-row smoke epochs scaled linearly to 7495 rows and 120 total epochs; model load and Drive overhead remain uncertain"}
coca_run.write_json_atomic(VALIDATION_DIR / "validation_record.json", record)
coca_run.write_json_atomic(VALIDATION_RECORD, record)
assert not (COCA_ROOT / "formal" / "_RUNNING.json").exists()
print(json.dumps(record, indent=2))
print("VALIDATION PASSED")
print("formal_training_started=false")